In [1]:
%%capture

import warnings
warnings.filterwarnings('ignore')

import altair as alt
import gcsfs
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

from calitp_portfolio import magics
from snapshot_utils import prep_data_utils, _color_palette
from snapshot_utils.project_vars import GCS_FILE_PATH

alt.data_transformers.enable("vegafusion")

In [2]:
#parameters cell
rtpa = "Sacramento Area Council of Governments"

In [3]:
%%capture_parameters
rtpa

{"rtpa": "Sacramento Area Council of Governments"}


# {rtpa}
## NTD Transit Performance Metrics

The UCLA Institute of Transportation Studies (UCLA ITS) suggests that:
>Updating the policy and legislation that governs state transit funding could help make expenditures more effective and better aligned with the state’s goals of VMT and GHG reduction, which transit can achieve only through increased ridership.

The UCLA ITS recommends using cost-efficiency metrics (operating expense per VRM/VRH/UPT) and service-effectiveness metrics (passenters per VRM/VRH) to compare transit-oriented vs. auto-oriented markets. 

The charts below display these metrics by different categories.

## Performance Metrics Explained

| Metric type          | Metric example                  | Implicit Goal(s)                       | Advantages                                   | Limitations                                  |
|----------------------|---------------------------------|---------------------------------------|----------------------------------------------|----------------------------------------------|
| Cost-efficiency     | Operating cost per revenue hour (opex_per_vrh) | Reduce costs                         | Useful in both financial and service planning | Favors high labor productivity in dense, congested areas; does not track use |
|                      | Operating cost per revenue mile (opex_per_vrm) |                                       |                                              |                                              |
|                      | Operating cost per vehicle trip (opex_per_upt) |                                       |                                              |                                              |
| Service-effectiveness| Passengers per revenue-vehicle hour (upt_per_vrh) | Increase ridership; reduce poorly patronized service | Useful for service planning; emphasizes what matters to riders | Favors high ridership; does not track costs   |
|                      | Passengers per revenue-vehicle mile (upt_per_vrm) | Increase ridership; reduce low-ridership route miles/segments | Useful for service planning                | Favors high ridership and fast vehicle speeds; does not track costs |


In [4]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    # should only certain columns be read in? now this table is much larger
    filesystem=gcsfs.GCSFileSystem(),
    columns = [
        "ntd_id", "source_agency", "agency_status", "source_city", 
        "year",
        "mode", "mode_full_name", "type_of_service", "type_of_service_full_name",
        "reporter_type", "reporting_module", "source_state", "primary_uza_name",
        "unlinked_passenger_trips", "vehicle_revenue_hours", "vehicle_revenue_miles",
        "operating_expenses_total",
        "opex_per_vrh", "opex_per_vrm", "opex_per_upt", 
        "upt_per_vrh", "upt_per_vrm",
        "farebox_recovery_ratio", "fare_revenue",
    ]
).pipe(
    prep_data_utils.merge_with_crosswalk
).query(
    f'rtpa_name == "{rtpa}"'
).dropna(
    subset="unlinked_passenger_trips"
).rename(columns = {"source_agency": "agency"})

## Agency

In [5]:
agency_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["ntd_id", "agency", "year", "rtpa_name"]
)

In [6]:
agency_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ntd_id                    56 non-null     object 
 1   agency                    56 non-null     object 
 2   year                      56 non-null     Int64  
 3   rtpa_name                 56 non-null     object 
 4   unlinked_passenger_trips  56 non-null     Int64  
 5   vehicle_revenue_miles     56 non-null     Int64  
 6   vehicle_revenue_hours     56 non-null     Int64  
 7   operating_expenses_total  56 non-null     Int64  
 8   opex_per_upt              56 non-null     Float64
 9   opex_per_vrh              56 non-null     Float64
 10  opex_per_vrm              56 non-null     Float64
 11  upt_per_vrh               56 non-null     Float64
 12  upt_per_vrm               56 non-null     Float64
dtypes: Float64(5), Int64(5), object(3)
memory usage: 6.4+ KB


In [7]:
agency_df.head(2)

,ntd_id,agency,year,rtpa_name,unlinked_passenger_trips,vehicle_revenue_miles,vehicle_revenue_hours,operating_expenses_total,opex_per_upt,opex_per_vrh,opex_per_vrm,upt_per_vrh,upt_per_vrm
0,90019,Sacramento Regional Transit District,2018,Sacramento Area Council of Governments,20890308,10705945,807817,152660649,7.31,188.98,14.26,25.86,1.95
1,90019,Sacramento Regional Transit District,2019,Sacramento Area Council of Governments,19989131,10989944,824189,166763517,8.34,202.34,15.17,24.25,1.82


In [8]:
# set some chart variables
color_scale = _color_palette.CALITP_CATEGORY_BRIGHT_COLORS + _color_palette.CALITP_CATEGORY_BOLD_COLORS

WIDTH = 400
HEIGHT = 250

In [9]:
def readable(word: str) -> str:
    """
    Coerce words that are abbreviated into readable labels for viz.
    """
    ABBREV_TO_FULL = {
        "opex_per_upt": "Operating Expense per Unlinked Passenger Trip",
        "opex_per_vrh": "Operating Expense per Vehicle Revenue Hour",
        "opex_per_vrm": "Operating Expense per Vehicle Revenue Mile",
        "upt_per_vrh": "Unlinked Passenger Trips per Vehicle Revenue Hour",
        "upt_per_vrm": "Unlinked Passenger Trips per Vehicle Revenue Mile",
    }

    if word in ABBREV_TO_FULL.keys():
        word = ABBREV_TO_FULL[word]

    return word.replace("_full_name", "").replace("_", " ").title().replace("Of", "of")

In [10]:
# Define all shared chart functions here
def title_by_group(
    group_col: str, 
    x_col: str = None,
    y_col: str = None,
):
    """
    Set title here for consistency.
    """
    readable_group = readable(group_col)
    
    if group_col == "reporter_type":
        readable_group = f"NTD {readable_group}"

    # For scatterplots
    if x_col and y_col:
        return f"Scatter: {readable(y_col)} by {readable(x_col)}"
    # For line plots by year
    if not x_col and y_col: 
        return f"{readable(y_col)} by {readable_group}"
    else:
        return readable_group
        
def tooltip_by_group(group_col: str): 
    """
    Consistent set of tooltip columns.
    """
    return ["year", group_col, "opex_per_upt", "opex_per_vrh", "opex_per_vrm", "upt_per_vrh", "upt_per_vrm", "rtpa_name"]


In [11]:
def make_base_chart(
    df: pd.DataFrame,
    y_col: str,
    color_col: str,
) -> alt.Chart:
    """
    Use 1 base chart function. 
    year is always x-axis, make it ordinal for better display.
    tooltip is standardized with function to populate as much as we can.

    Everything else, such as title, even .mark_line(), .mark_bar() 
    can be layered on top of this function.

    This one is similar to annual report.
    """
    selection = alt.selection_point(fields=[color_col], bind='legend')

    chart = (
        alt.Chart(df)
        .encode(
            x=alt.X("year:O"),    
            y=alt.Y(
                y_col, title=y_col, 
                scale=alt.Scale(zero=False, clamp=True)
            ),
            color=alt.Color(
                color_col,
                scale=alt.Scale(range=color_scale),
                #legend=None
            ),
            opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.02)),
            tooltip=tooltip_by_group(color_col),
        ).properties(width=WIDTH, height=HEIGHT)
        .interactive()
    ).add_params(selection)

    return chart

In [12]:
def combined_line_charts_by_year(
    df, 
    color_col: str
):
    """
    Cost-efficiency and service-effectiveness charts
    can be titled, labeled separately, but ultimately,
    want to display as 2 groups.
    Group them together here, and add the legend selector again,
    it loses the legend selector by the end.
    """
    cost_efficiency_chart_list = [
        make_base_chart(
            df, 
            y_col = m,
            color_col = color_col
        ).mark_line(
            point=alt.OverlayMarkDef(filled=False, fill="white")
        ).properties(
            title=title_by_group(color_col, x_col= None, y_col = m)
        )
        for m in cost_efficiency_metrics
    ]

    # Loop over service-effectiveness metrics and create charts by agency
    service_effectiveness_chart_list = [
        make_base_chart(
            df, 
            y_col = m,
            color_col = color_col
        ).mark_line(
            point=alt.OverlayMarkDef(filled=False, fill="white")
        ).properties(
            title=title_by_group(color_col, x_col= None, y_col = m)
        )
        for m in service_effectiveness_metrics
    ]

    selection = alt.selection_point(fields=[color_col], bind='legend')
    
    chart1 = alt.hconcat(*cost_efficiency_chart_list).properties(
        title=f"Cost Efficiency by {readable(color_col)}"
    )
    
    chart2 = alt.hconcat(*service_effectiveness_chart_list).properties(
        title=f"Service Effectiveness by {readable(color_col)}"
    )

    combined_chart = alt.vconcat(chart1, chart2).add_params(selection) # add legend selector here again

    return combined_chart 


In [13]:
# Loop over the cost-efficiency metrics and create charts by agency
cost_efficiency_metrics = ["opex_per_vrh", "opex_per_vrm", "opex_per_upt"]
service_effectiveness_metrics = ["upt_per_vrh", "upt_per_vrm"]

combined_line_charts_by_year(agency_df, "agency")

alt.VConcatChart(...)

## Mode

In [14]:
mode_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["mode", "mode_full_name", "year", "rtpa_name"]
)

In [15]:
combined_line_charts_by_year(mode_df, "mode_full_name")

alt.VConcatChart(...)

In [16]:
# scatterplot with all the years and agencies?
# The regression lines might indicate marginal cost of providing that service
# steeper lines show higher marginal cost?
def make_scatterplot_with_line(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    color_col: str,
) -> alt.Chart:
    """
    Add scatterplot with regression line.
    Since scatterplot is log-log, the regression line we use 
    can be linear.
    Only upt is not log(upt).
    If we did regular_x, regular_y, we would have to use power curve?
    """
    scatter_selection = alt.selection_point(fields=[color_col], bind='legend')

    scatter = alt.Chart(df).mark_point().encode(
        y=alt.Y(y_col, title = f"log({readable(y_col)})").scale(type="log"),
        color=alt.Color(
            color_col, 
            title=readable(color_col),
            scale=alt.Scale(range=color_scale)
        ),
        tooltip = [color_col, "year", "ntd_id",
                   x_col, y_col, "rtpa_name"],
        opacity=alt.when(scatter_selection).then(alt.value(1)).otherwise(alt.value(0.02)),
    )

    # Be explicit here around which ones need log scale (exceptions listed out, just upt)
    if x_col == "unlinked_passenger_trips":
        scatter = scatter.encode(
            x=alt.X(x_col, title=readable(x_col))
        )
    else:
        scatter = scatter.encode(
            x=alt.X(x_col, title=f"log({readable(x_col)})").scale(type="log")
        )
    
    scatter_line = scatter.transform_regression(
        regression=x_col, 
        on=y_col,
        method="exp" if x_col == "unlinked_passenger_trips" else "pow",
        groupby=[color_col]
    ).mark_line()

    chart = (scatter_line + scatter).properties(
        width=WIDTH, height=HEIGHT,
        title = title_by_group(color_col, x_col, y_col)
    ).add_params(scatter_selection)
    
    return chart

### Cost-efficiency metrics
Cost-efficiency measures inputs to outputs: For example, the cost of operating an hour of transit service.

Per the UCLA ITS Paper
>Transit-oriented markets (which are predominantly urban), transit service tends to be relatively service-effective. But high operating costs on these (mostly) older, larger systems can inhibit efforts to improve ridership by adding service. In such contexts, assessing systems with an emphasis on **cost-efficiency (i.e., the cost of operating an hour of service)** grounds would provide incentives for agencies to **manage their costs** so as to be able to provide more service with available funding.

### Service-effectiveness metrics
Service-effectiveness measures outputs to consumption: For example, passenger boardings per service hour.

Per the UCLA ITS Paper
>[In] more auto-oriented markets, transit operators tend to be relatively cost-efficient, in that they have lower operating costs but serve fewer riders. In this context, assessing systems with an emphasis on **service-effectiveness (i.e., passenger boardings per service hour)** will motivate operators to **improve ridership** by changing service hours, routes, and fares to better match local demand. Agencies might also implement fare programs with schools and other institutions, and even work with municipalities on improving land use around transit in order to increase the relative attractiveness of transit service.

### SA NOTES ON YEAR POOLING

Right now the axes just look log-log because of the scale setting, but transform_regression still fits the line to the raw, un-logged numbers. For most panels, method="pow" fixes this since both axes are logged. But the UPT-as-x panel only has the y-axis logged, not the x-axis, so the straight-line fit that matches that combination is an exponential curve, not a power curve,  that's what method="exp" gives us. Using pow or linear there would still draw curved, and the slope wouldn't mean what we want.

In [17]:
cost_efficiency_scatter_list = [
    make_scatterplot_with_line(
        df, 
        m, 
        "operating_expenses_total", 
        "mode_full_name",
    ) 
    for m in ["unlinked_passenger_trips", "vehicle_revenue_hours", "vehicle_revenue_miles"]
]

cost_efficiency_scatter = alt.hconcat(*cost_efficiency_scatter_list).resolve_scale(
    x='independent', y='shared'
)

service_effectiveness_scatter_list = [
    make_scatterplot_with_line(
        df, 
        m, 
        "unlinked_passenger_trips", 
        "mode_full_name",
    ) 
    for m in ["vehicle_revenue_hours", "vehicle_revenue_miles"]
]


service_effectiveness_scatter = alt.hconcat(
    *service_effectiveness_scatter_list
).resolve_scale(
    x='independent', y='shared'
)

alt.vconcat(cost_efficiency_scatter, service_effectiveness_scatter).interactive()

alt.VConcatChart(...)

### SA NOTES

For each transit mode, has the relationship between service and cost/ridership stayed the same across the years? 
- This will confirm whether or not we can pool (it might look the same but lets fit one regression per mode that lets the slope shift by year, then test whether those year-to-year shifts are big enough to matter statistically.
- Does letting the slope be different for each year actually explain meaningfully more than just using one slope for everything?

In [18]:
def check_poolability(df, x_col, y_col, min_obs_per_year = 3):
    df_check = df.copy()
    if x_col == "unlinked_passenger_trips": # This is to match the fact that upt isnt logged 
        df_check["log_x"] = df_check[x_col]
    else:
        df_check["log_x"] = np.log(df_check[x_col])
        
    df_check["log_y"] = np.log(df_check[y_col])
    df_check["year"] = df_check["year"].astype(str)

    # drop years with too few agencies to estimate a slope
    counts = df_check["year"].value_counts()
    df_check = df_check[df_check["year"].isin(counts[counts >= min_obs_per_year].index)]

    model = smf.ols("log_y ~ log_x * year", data = df_check).fit()
    terms = [p for p in model.params.index if "log_x:year" in p]
    if not terms:
        print("No interaction terms — can't test.")
        return

    result = model.f_test([f"{t} = 0" for t in terms])

    print(f"p = {result.pvalue:.4f}  →  {'pool OK' if result.pvalue > 0.05 else 'DO NOT pool'}")

In [19]:
bus_df = df[df["mode_full_name"] == "Bus"]
check_poolability(bus_df, "vehicle_revenue_hours", "operating_expenses_total")

p = 0.9701  →  pool OK


In [20]:
bus_df = df[df["mode_full_name"] == "Bus"]
check_poolability(bus_df, "unlinked_passenger_trips", "operating_expenses_total")

p = 0.8086  →  pool OK


In [21]:
dr_df = df[df["mode_full_name"] == "Demand Response"]
check_poolability(dr_df, "unlinked_passenger_trips", "operating_expenses_total")

p = 0.9641  →  pool OK


In [22]:
dr_df = df[df["mode_full_name"] == "Demand Response"]
check_poolability(dr_df, "vehicle_revenue_hours", "operating_expenses_total")

p = 0.0048  →  DO NOT pool


I think we can run this for each mode and x_value_y, value pair and then decide whether we can pool or not. The model fits a seperate slope for each year and then F test checks whether that extra flexibility actually improves the fit compared to forcing all years to share one slope. If it barely helps- the years are basically the same, F value is small and p value is high. If it helps a lot the years are behaving differently f-value comes out large and p-value is low.

For Demand Response, the VRH vs UPT relationship is stable across years, but the VRH vs Operating Expense relationship is not, likely due to something like inflation, COVID.

### SA Notes on Marginal Cost 

Once we use pow,  for log-log the slope we get is elasticity, for every 1% increase in service what percent increase in cost do we get? 
example : if bus has 0.9 slope while rail as 1.3 it would mean that bus cost grow slightly slower than service scales up while rail cost would grow faster. 

Marginal cost is for one or more vehicle revenue hour, how many dollars does it cost so it is dollar-per-unit number rather than a percentage could come from a raw-value relationship rather han log or power curve.

### SA Notes on what plots to use if we can pool the years 

Where pooling checks out, we can add an elasticity forest plot alongside the existing pooled-year scatterplots for the log-log variables (vehicle revenue hours, vehicle revenue miles). The scatterplots show every individual agency/year observation with a regression line per mode, while the forest plot summarizes each of those regressions into a single elasticity estimate with its confidence interval,  easier to compare across modes. For the cost-efficiency regressions, a reference line at 1.0 marks proportional scaling: below 1 indicates economies of scale (costs grow slower than service), above 1 indicates diseconomies of scale (costs grow faster than service).

This doesn't apply to the unlinked-passenger-trips panel, since UPT isn't log-scaled as x — its regression slope is a semi-elasticity, not a true elasticity, so it can't be shown on the same axis or interpreted the same way.

In [23]:
def calculate_elasticities(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    color_col: str,
) -> pd.DataFrame:
    results = []
    for mode, group in df.groupby(color_col):
        data = group[[x_col, y_col]].copy()
        data[x_col] = pd.to_numeric(data[x_col], errors="coerce")
        data[y_col] = pd.to_numeric(data[y_col], errors="coerce")
        data = data.dropna()
        data = data[(data[x_col] > 0) & (data[y_col] > 0)]

        # match the chart: UPT-as-x stays linear (exp fit), everything else gets logged (pow fit)
        if x_col == "unlinked_passenger_trips":
            x_input = data[x_col].to_numpy(dtype=float)
        else:
            x_input = np.log(data[x_col].to_numpy(dtype=float))

        log_y = np.log(data[y_col].to_numpy(dtype=float))

        X = np.column_stack([np.ones(len(x_input)), x_input])
        model = sm.OLS(log_y, X).fit()
        coef = model.params[1]
        ci_low, ci_high = model.conf_int()[1]
        results.append({
            color_col: mode,
            "elasticity" if x_col != "unlinked_passenger_trips" else "semi_elasticity": coef,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "n": len(data),
        })
    return pd.DataFrame(results)

In [24]:
def elasticity_forest_plot(
    elasticity_df: pd.DataFrame,
    color_col: str,
    title: str = "Cost elasticity by mode",
) -> alt.Chart:
    points = alt.Chart(elasticity_df).mark_point(
        filled=True,
        size=100
    ).encode(
        x=alt.X("elasticity:Q", title="Cost elasticity", scale=alt.Scale(zero=False)),
        y=alt.Y(f"{color_col}:N", title=None, sort="-x"),
        tooltip=[
            alt.Tooltip(f"{color_col}:N", title="Mode"),
            alt.Tooltip("elasticity:Q", title="Elasticity", format=".2f"),
            alt.Tooltip("ci_low:Q", title="95% CI low", format=".2f"),
            alt.Tooltip("ci_high:Q", title="95% CI high", format=".2f"),
            alt.Tooltip("n:Q", title="N"),
        ],
    )

    ci = alt.Chart(elasticity_df).mark_rule(strokeWidth=3).encode(
        x="ci_low:Q",
        x2="ci_high:Q",
        y=f"{color_col}:N",
    )

    reference = alt.Chart(pd.DataFrame({"elasticity": [1]})).mark_rule(
        color="gray", strokeDash=[5, 5], strokeWidth=2
    ).encode(x="elasticity:Q")

    return (reference + ci + points).properties(
        width=600,
        height=max(250, len(elasticity_df) * 45),
        title=title
    )


In [25]:
vrh_elasticities = calculate_elasticities(
    df, "vehicle_revenue_hours", "operating_expenses_total", "mode_full_name"
)
vrm_elasticities = calculate_elasticities(
    df, "vehicle_revenue_miles", "operating_expenses_total", "mode_full_name"
)

vrh_forest = elasticity_forest_plot(
    vrh_elasticities, "mode_full_name",
    title="Cost elasticity by mode (vehicle revenue hours)"
)
vrm_forest = elasticity_forest_plot(
    vrm_elasticities, "mode_full_name",
    title="Cost elasticity by mode (vehicle revenue miles)"
)

alt.hconcat(vrh_forest, vrm_forest)

alt.HConcatChart(...)

Bus shows a reliable, precisely estimated elasticity close to 1.0. Commuter Bus and Light Rail have confidence intervals too wide to trust as point estimates, likely due to small sample sizes. Demand Response is the trickiest case: its confidence interval looks just as tight as Bus's, but a separate test confirmed that pooling years together isn't valid for this mode's cost relationship.

Same can be done for Type of Service as well. 

## Type of Service

In [ ]:
tos_df = prep_data_utils.calculate_efficiency_metrics_by_group(
    df, 
    ["type_of_service", "type_of_service_full_name", "year", "rtpa_name",]
)

In [ ]:
combined_line_charts_by_year(tos_df, "type_of_service_full_name")

In [ ]:
cost_efficiency_scatter_list = [
    make_scatterplot_with_line(
        df, 
        m, 
        "operating_expenses_total", 
        "type_of_service_full_name",
    )
    for m in ["vehicle_revenue_hours", "vehicle_revenue_miles"]
]

cost_efficiency_scatter = alt.hconcat(*cost_efficiency_scatter_list).resolve_scale(
    x='independent', y='shared'
)

service_effectiveness_scatter_list = [
    make_scatterplot_with_line(
        df, 
        m, 
        "unlinked_passenger_trips", 
        "type_of_service_full_name",
    )
    for m in ["vehicle_revenue_hours", "vehicle_revenue_miles"]
]


service_effectiveness_scatter = alt.hconcat(*service_effectiveness_scatter_list).resolve_scale(
    x='independent', y='shared'
)

alt.vconcat(cost_efficiency_scatter, service_effectiveness_scatter)